In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [4]:
datasetName = "Adult"

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

dataset = pd.read_csv(f"{ugce_dir}/data/adult.csv")
TARGET_COLUMN = 'target'
dataset[TARGET_COLUMN] = LabelEncoder().fit_transform(dataset[TARGET_COLUMN])
target = dataset[TARGET_COLUMN]
datasetX = dataset.drop(columns=[TARGET_COLUMN])

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = datasetX.columns.to_list()
categorical = x_train.columns.difference(numerical)

try:
    import joblib
    model = joblib.load(f"{ugce_dir}/results/models/{datasetName}_model.pkl")
except:
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    transformations = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical),
            ('cat', categorical_transformer, categorical)])
    
    model = RandomForestClassifier(random_state=42)

    model = Pipeline(steps=[('preprocessor', transformations),
                        ('classifier', model)])
    model.fit(x_train, y_train)

    import joblib
    os.makedirs(f"{ugce_dir}/results/models", exist_ok=True)
    joblib.dump(model, f"{ugce_dir}/results/models/{datasetName}_model.pkl")

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances
print("Number of instances to explain: ", len(instances_to_explain))

Accuracy:  0.8469648889343843
Number of instances to explain:  2063


In [6]:
feat_unique = {}
for col in datasetX.columns:
    feat_unique[col] = len(datasetX[col].unique())
feat_unique
## sort the features by the number of unique values
top_3_difficult_to_change_cols = sorted(feat_unique.items(), key=lambda x: x[1])
top_3_difficult_to_change_cols += ['age']
print(f"Top 3 difficult to change columns: {top_3_difficult_to_change_cols[:2]}")
features_to_vary = [col for col in datasetX.columns if col not in top_3_difficult_to_change_cols]
features_to_vary

Top 3 difficult to change columns: [('sex', 2), ('race', 5)]


['workclass',
 'education',
 'education-num',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'capital-gain',
 'capital-loss',
 'hours-per-week']

In [7]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

In [8]:
numerical_columns = iea.dataset.select_dtypes(include=['int64', 'float64']).columns

non_zero_descriptions = {}

for col in numerical_columns:
    non_zero_values = iea.dataset[iea.dataset[col] != 0][col]
    if not non_zero_values.empty:
        non_zero_descriptions[col] = non_zero_values.describe()

for feature, stats in non_zero_descriptions.items():
    print(f"\n Feature: {feature}")
    print(stats)


 Feature: age
count    48842.000000
mean        38.643585
std         13.710510
min         17.000000
25%         28.000000
50%         37.000000
75%         48.000000
max         90.000000
Name: age, dtype: float64

 Feature: workclass
count    46043.000000
mean         4.105727
std          1.143796
min          1.000000
25%          4.000000
50%          4.000000
75%          4.000000
max          8.000000
Name: workclass, dtype: float64

 Feature: education
count    47453.000000
mean        10.589573
std          3.501708
min          1.000000
25%          9.000000
50%         11.000000
75%         12.000000
max         15.000000
Name: education, dtype: float64

 Feature: education-num
count    48842.000000
mean        10.078089
std          2.570973
min          1.000000
25%          9.000000
50%         10.000000
75%         12.000000
max         16.000000
Name: education-num, dtype: float64

 Feature: marital-status
count    42209.000000
mean         3.030278
std          1.176

In [9]:
iea.numerical_columns

['age',
 'workclass',
 'education',
 'education-num',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'capital-gain',
 'capital-loss',
 'hours-per-week']

# Constraints Type Series:
1. Immutability
2. Ranges
3. Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'sex': 'i',
        'race': 'i'
    },
    2: {
        'sex': 'i',
        'race': 'i',    
        'hours-per-week': (1, 40),
        'capital-gain': (12000, 99999)
    },
    3: {
        'sex': 'i',
        'race': 'i',
        'hours-per-week': (1, 40),
        'capital-gain': (12000, 99999),
        'age': 'incr',
        'capital-loss': 'decr'
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

results_incremental_imm_ranges_direct_arr = []
import time
strategy = "fix_population_update_fitness"
for i in range(5):
    results_incremental_imm_ranges_direct = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_imm_ranges_direct_arr.append(results_incremental_imm_ranges_direct)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_imm_ranges_direct_arr, open(f"{results_dir}/results_incremental_imm_ranges_direct_arr.pkl", "wb"))

100%|██████████| 2063/2063 [14:37<00:00,  2.35it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [15:05<00:00,  2.28it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [15:41<00:00,  2.19it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [15:16<00:00,  2.25it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [14:42<00:00,  2.34it/s]


Empty intermediate counter: 0


In [ ]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_imm_ranges_direct = pickle.load(open(f'{results_dir}/results_incremental_imm_ranges_direct_arr.pkl', 'rb'))

# Constraints Type Series:
1. Ranges
2. Immutability
3. Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'hours-per-week': (1, 40),
        'capital-gain': (12000, 99999)
    },
    2: {
        'hours-per-week': (1, 40),
        'capital-gain': (12000, 99999),
        'sex': 'i',
        'race': 'i',    
        
    },
    3: {
        'hours-per-week': (1, 40),
        'capital-gain': (12000, 99999),
        'sex': 'i',
        'race': 'i',
        'age': 'incr',
        'capital-loss': 'decr'
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

results_incremental_ranges_imm_incr_arr = []
import time
strategy = "fix_population_update_fitness"
for i in range(5):
    results_incremental_ranges_imm_incr = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_ranges_imm_incr_arr.append(results_incremental_ranges_imm_incr)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_ranges_imm_incr_arr, open(f"{results_dir}/results_incremental_ranges_imm_incr_arr.pkl", "wb"))

100%|██████████| 2063/2063 [14:32<00:00,  2.37it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [13:48<00:00,  2.49it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [14:30<00:00,  2.37it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [14:35<00:00,  2.36it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [14:40<00:00,  2.34it/s]


Empty intermediate counter: 0


In [ ]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_ranges_imm_direct = pickle.load(open(f'{results_dir}/results_incremental_ranges_imm_incr_arr.pkl', 'rb'))

# Constraints Type Series:
1. Directionality
2. Immutability
3. Ranges

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'age': 'incr',
        'capital-loss': 'decr'
    },
    2: {
        'age': 'incr',
        'capital-loss': 'decr',
        'sex': 'i',
        'race': 'i', 
    },
    3: {
        'age': 'incr',
        'capital-loss': 'decr',
        'sex': 'i',
        'race': 'i',
        'hours-per-week': (1, 40),
        'capital-gain': (12000, 99999),
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

import time
strategy = "fix_population_update_fitness"
results_incremental_dir_im_range_arr = []
for i in range(5):
    results_incremental_dir_im_range = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_dir_im_range_arr.append(results_incremental_dir_im_range)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_dir_im_range_arr, open(f"{results_dir}/results_incremental_dir_im_range_arr.pkl", "wb"))

100%|██████████| 2063/2063 [14:07<00:00,  2.43it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [13:52<00:00,  2.48it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [14:50<00:00,  2.32it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [14:11<00:00,  2.42it/s]


Empty intermediate counter: 0


100%|██████████| 2063/2063 [15:10<00:00,  2.26it/s]


Empty intermediate counter: 0


In [ ]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_dir_im_range = pickle.load(open(f'{results_dir}/results_incremental_dir_im_range_arr.pkl', 'rb'))

# Make Plots

In [ ]:
from test_utils import gather_results_sequence_of_type_constraints
import matplotlib.pyplot as plt

constraint_orders = ["I→R→D", "R→I→D", "D→I→R"]

time_dynamic_imm_ranges_direct, avg_generations_imm_ranges_direct, avg_cfes_found_imm_ranges_direct, avg_proximity_loss_imm_ranges_direct, avg_sparsity_imm_ranges_direct, avg_intermediate_imm_ranges_incr, \
time_dynamic_ranges_imm_incr, avg_generations_ranges_imm_incr, avg_cfes_found_ranges_imm_incr, avg_proximity_loss_ranges_imm_incr, avg_sparsity_ranges_imm_incr, avg_intermediate_ranges_imm_incr, \
time_dynamic_dir_im_range, avg_generations_dir_im_range, avg_cfes_found_dir_im_range, avg_proximity_loss_dir_im_range, avg_sparsity_dir_im_range, avg_intermediate_dir_im_range =\
    gather_results_sequence_of_type_constraints(iea, results_incremental_imm_ranges_direct_arr, results_incremental_ranges_imm_incr_arr, results_incremental_dir_im_range_arr, verbose=True) 

cfe_found = [
        avg_cfes_found_imm_ranges_direct,
        avg_cfes_found_ranges_imm_incr,
        avg_cfes_found_dir_im_range
]
avg_time = [
    time_dynamic_imm_ranges_direct,
    time_dynamic_ranges_imm_incr,
    time_dynamic_dir_im_range
]
avg_weighted_l1 = [
    avg_proximity_loss_imm_ranges_direct,
    avg_proximity_loss_ranges_imm_incr,
    avg_proximity_loss_dir_im_range
]

avg_sparsity = [
    avg_sparsity_imm_ranges_direct,
    avg_sparsity_ranges_imm_incr,
    avg_sparsity_dir_im_range
]
results = {
    "cfe_found": cfe_found,
    "avg_time": avg_time,
    "avg_weighted_l1": avg_weighted_l1,
    "avg_sparsity": avg_sparsity
}

In [ ]:
table_data = pd.DataFrame(results, index=constraint_orders)
print(table_data.to_latex(float_format="%.4f"))

\begin{tabular}{lrrrr}
\toprule
 & cfe_found & avg_time & avg_weighted_l1 & avg_sparsity \\
\midrule
I→R→D & 79.8741 & 7.6252 & 0.0336 & 0.0198 \\
R→I→D & 80.0387 & 7.2802 & 0.0335 & 0.0198 \\
D→I→R & 80.1065 & 7.3241 & 0.0296 & 0.0186 \\
\bottomrule
\end{tabular}

